# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [3]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 12) (142,)


## 2. Pipeline de procesamiento

In [4]:
feature_columns = contract["features"]

# TODO: crea un preprocesador para árboles con imputación por mediana
# TODO: crea otro con imputación y StandardScaler
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("imputer", SimpleImputer(strategy="median"), feature_columns)
    ]
)

scaler_preprocessor = ColumnTransformer(
    transformers=[
        ("imputer", SimpleImputer(strategy="median"), feature_columns),
        ("scaler", StandardScaler(), feature_columns)
    ]
)


**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Escribe aquí tu respuesta.

## 3. Modelos candidatos

In [6]:
# TODO: crea pipelines para DecisionTree, LogisticRegression y KNN
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", scaler_preprocessor),
        ("classifier", LogisticRegression(random_state=RANDOM_STATE))
    ]
)

decision_tree_pipeline = Pipeline(
    steps=[
        ("preprocessor", tree_preprocessor),
        ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE))
    ]
)

knn_pipeline = Pipeline(
    steps=[
        ("preprocessor", scaler_preprocessor),
        ("classifier", KNeighborsClassifier())
    ]
)

## 4. Comparación inicial con validación cruzada

In [9]:
# TODO: usa StratifiedKFold de 5 folds y cross_validate
# Reporta accuracy promedio de train, accuracy promedio de validation y desviación estándar
# Guarda el resumen en initial_comparison
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
pipelines = {
    "Logistic Regression": logistic_pipeline,
    "Decision Tree": decision_tree_pipeline,
    "KNN": knn_pipeline
}

initial_comparison = {}
for name, pipeline in pipelines.items():
    scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring="accuracy", return_train_score=True)
    initial_comparison[name] = {
        "train_accuracy": scores["train_score"].mean(),
        "validation_accuracy": scores["test_score"].mean(),
        "std": scores["test_score"].std()
    }


c:\Users\pablo\Documents\SI003_IA_PABLO_ARANGO\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\pablo\Documents\SI003_IA_PABLO_ARANGO\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-le

## 5. Optimización de hiperparámetros

In [10]:
# TODO: define espacios pequeños de búsqueda para los tres modelos
# TODO: ejecuta GridSearchCV con scoring="accuracy"
# Guarda las búsquedas en searches y el resumen en tuned_comparison
param_grids = {
    "Logistic Regression": {
        "classifier__C": [0.01, 0.1, 1, 10]
    },
    "Decision Tree": {
        "classifier__max_depth": [None, 5, 10, 15],
        "classifier__min_samples_split": [2, 5, 10]
    },
    "KNN": {
        "classifier__n_neighbors": [3, 5, 7, 9],
        "classifier__weights": ["uniform", "distance"]
    }
}
grid_searches = {}
for name, param_grid in param_grids.items():
    grid_search = GridSearchCV(pipelines[name], param_grid, scoring="accuracy", cv=cv)
    grid_search.fit(X_train, y_train)
    grid_searches[name] = grid_search
tuned_comparison = {}
for name, grid_search in grid_searches.items():
    tuned_comparison[name] = {
        "best_params": grid_search.best_params_,
        "best_score": grid_search.best_score_
    }

c:\Users\pablo\Documents\SI003_IA_PABLO_ARANGO\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\pablo\Documents\SI003_IA_PABLO_ARANGO\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-le

## 6. Selección y persistencia del campeón

In [15]:
# TODO: selecciona el mayor best_score_ de CV
# champion_name = ...
# champion = ...

# TODO: guarda champion_model.joblib, training_metadata.json y cv_results.csv
champion_name = max(tuned_comparison, key=lambda name: tuned_comparison[name]["best_score"])
champion = grid_searches[champion_name].best_estimator_
champion_model_path = ARTIFACTS_DIR / "champion_model.joblib"
joblib.dump(champion, champion_model_path)
training_metadata = {
    "champion": champion_name,
    "best_params": tuned_comparison[champion_name]["best_params"],
    "best_score": tuned_comparison[champion_name]["best_score"]
}
with open(ARTIFACTS_DIR / "training_metadata.json", "w") as f:
    json.dump(training_metadata, f)
cv_results_df = pd.DataFrame(grid_searches[champion_name].cv_results_)
cv_results_df.to_csv(REPORTS_DIR / "cv_results.csv", index=False)

In [16]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

> Explica la decisión considerando desempeño, variabilidad, sobreajuste, complejidad e interpretabilidad.

## Checklist

- [ ] Todo el procesamiento está dentro de pipelines.
- [ ] Comparé tres familias de modelos.
- [ ] Optimicé hiperparámetros con CV.
- [ ] Elegí el campeón sin usar test.
- [ ] Guardé el pipeline completo y sus metadatos.